In [2]:
import numpy as np
import cv2
import pickle
import os

# Load Phase 1 results
with open('phase1_results.pkl', 'rb') as f:
    p1 = pickle.load(f)

img = p1['img']
height, width = p1['height'], p1['width']
region_map = p1['region_map']
region_id = p1['region_id']
rsizes = p1['rsizes']

output_dir = "comic/phase2_output"
os.makedirs(output_dir, exist_ok=True)

print(f"Phase 1 data loaded")
print(f"Image: {height}x{width}, Regions: {region_id}")

# Save original image
original_path = os.path.join(output_dir, "00_original_image.jpg")
cv2.imwrite(original_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

Phase 1 data loaded
Image: 393x800, Regions: 38238


True

In [3]:
def detect_text_regions(img, region_map):
    """
    Detect text regions based on:
    - Dark color (Y < 100)
    - Connected components
    """
    
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    
    # Binary threshold for dark regions (text is usually dark)
    _, binary = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY_INV)
    
    # Morphological operations to connect nearby text pixels
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=1)
    
    return binary

# Get text mask
text_mask = detect_text_regions(img, region_map)

# Find which regions are primarily text
text_regions = set()
unique_rids = np.unique(region_map[region_map >= 0])

for rid in unique_rids:
    region_mask = (region_map == rid)
    text_pixels_in_region = np.sum(text_mask[region_mask] > 0)
    total_pixels_in_region = np.sum(region_mask)
    
    # If >50% of region is text-like
    if total_pixels_in_region > 0 and text_pixels_in_region / total_pixels_in_region > 0.5:
        text_regions.add(int(rid))

print(f"Text regions detected: {len(text_regions)}")

# Visualize text regions
text_vis = img.copy()
for rid in text_regions:
    mask = (region_map == rid)
    text_vis[mask] = np.clip(text_vis[mask] * 0.5, 0, 255).astype(np.uint8)  # Darken

text_vis_path = os.path.join(output_dir, "01_text_regions_detected.jpg")
cv2.imwrite(text_vis_path, cv2.cvtColor(text_vis, cv2.COLOR_RGB2BGR))
print(f"Text visualization saved: {text_vis_path}")

Text regions detected: 8933
Text visualization saved: comic/phase2_output\01_text_regions_detected.jpg


In [4]:
def detect_edge_regions(img, region_map, rsizes, edge_threshold=50):
    """
    Detect thin edge regions (like character outlines)
    """
    
    edge_regions = set()
    unique_rids = np.unique(region_map[region_map >= 0])
    
    for rid in unique_rids:
        # Thin regions with small perimeter indicate edges
        if rsizes[rid] < edge_threshold and rsizes[rid] > 0:
            mask = (region_map == rid)
            
            # Check if region is mostly dark (edge/outline color)
            region_pixels = img[mask]
            avg_brightness = np.mean(cv2.cvtColor(region_pixels.reshape(1, -1, 3), cv2.COLOR_RGB2GRAY))
            
            if avg_brightness < 150:  # Dark color
                edge_regions.add(int(rid))
    
    return edge_regions

edge_regions = detect_edge_regions(img, region_map, rsizes)
print(f"Edge/Outline regions detected: {len(edge_regions)}")

# Combine text + edge regions
special_regions = text_regions | edge_regions
print(f"Total special regions (text + edge): {len(special_regions)}")

# Visualize all special regions
special_vis = img.copy()
for rid in special_regions:
    mask = (region_map == rid)
    special_vis[mask] = np.clip(special_vis[mask] * 0.7, 0, 255).astype(np.uint8)

special_vis_path = os.path.join(output_dir, "02_text_and_edge_regions.jpg")
cv2.imwrite(special_vis_path, cv2.cvtColor(special_vis, cv2.COLOR_RGB2BGR))
print(f"Special regions visualization saved: {special_vis_path}")

Edge/Outline regions detected: 10276
Total special regions (text + edge): 10848
Special regions visualization saved: comic/phase2_output\02_text_and_edge_regions.jpg


In [5]:
phase2_results = {
    'img': img,
    'height': height,
    'width': width,
    'region_map': region_map,
    'text_regions': text_regions,
    'edge_regions': edge_regions,
    'special_regions': special_regions
}

with open('phase2_results.pkl', 'wb') as f:
    pickle.dump(phase2_results, f)

print("Phase 2 results saved to phase2_results.pkl")
print(f"Key outputs:")
print(f"  - text_regions: {len(text_regions)}")
print(f"  - edge_regions: {len(edge_regions)}")
print(f"  - Total special regions: {len(special_regions)}")

Phase 2 results saved to phase2_results.pkl
Key outputs:
  - text_regions: 8933
  - edge_regions: 10276
  - Total special regions: 10848


In [6]:

# Text regions 
text_vis = img.copy().astype(float)
for rid in text_regions:
    mask = (region_map == rid)
    text_vis[mask] = [255, 0, 0]  

text_vis_path = os.path.join(output_dir, "01_text_regions_RED.jpg")
cv2.imwrite(text_vis_path, cv2.cvtColor(text_vis.astype(np.uint8), cv2.COLOR_RGB2BGR))

# Text + Edge 
special_vis = img.copy().astype(float)
for rid in special_regions:
    mask = (region_map == rid)
    special_vis[mask] = [255, 0, 0]  

special_vis_path = os.path.join(output_dir, "02_text_and_edge_RED.jpg")
cv2.imwrite(special_vis_path, cv2.cvtColor(special_vis.astype(np.uint8), cv2.COLOR_RGB2BGR))

print("Red visualization saved")

Red visualization saved
